In [1]:
import os
os.chdir("c:/users/prasa/chicken-disease")

%pwd

'c:\\users\\prasa\\chicken-disease'

In [2]:
from dataclasses import dataclass
from pathlib import Path
@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    basemodel: Path
    updated_basemodel: Path
    params_imagesize: list
    params_learning_rate: float
    params_include_top: bool
    params_weights: str
    params_classes: int


In [3]:

from src.chicken_project.constant import *
from src.chicken_project.utils.common import read_yaml,create_directories
class ConfigurationManager:
    def __init__(self,config_filepath=config_file_path,
                 params_filepath=params_file_path):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        create_directories([self.config.artifacts_roots])
    def get_PrepareBaseModelConfig(self)->PrepareBaseModelConfig:
        config=self.config.prepareBaseModel
        create_directories([config.root_dir])
        PreparebasemodelConfig= PrepareBaseModelConfig(root_dir= Path(config.root_dir),
                                                       basemodel= Path(config.basemodel),
                                                       updated_basemodel= Path(config.updated_basemodel),
                                                       params_imagesize= self.params.imagesize,
                                                       params_learning_rate= self.params.learning_rate,
                                                       params_include_top= self.params.include_top,
                                                       params_weights= self.params.weights,
                                                       params_classes= self.params.classes)
        
        return PreparebasemodelConfig


In [4]:
import os
import tensorflow as tf

class PrepareBaseModel:
    def __init__(self,config:PrepareBaseModelConfig):
        self.config=config
 

    def get_basemodel(self):
        self.model=tf.keras.applications.vgg16.VGG16(
        input_shape= self.config.params_imagesize,
        weights= self.config.params_weights,
        include_top=self.config.params_include_top
    )
        self.save_model(path=self.config.basemodel,model=self.model)
    @staticmethod
    def preparefullmodel(model,classes,freeze_all,freeze_till,learning_rate):
        if freeze_all:
            for layer in model.layers:
                model.trainable=False
        elif(freeze_till is not None ) and (freeze_till > 0):
           for layer in model.layer[: -freeze_till]:
                model.trainable=False
        flatten_in = tf.keras.layers.Flatten()(model.output)
        prediction = tf.keras.layers.Dense(
            units = classes,
            activation ="softmax")(flatten_in)
        full_model= tf.keras.Model(inputs=model.input,outputs=prediction)
        full_model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),loss=tf.keras.losses.CategoricalCrossentropy(),metrics=["accuracy"])
        full_model.summary()
        return full_model
    def update_basemodel(self):
        self.full_model = self.preparefullmodel(model=self.model,
                                            classes=self.config.params_classes,
                                            freeze_all=True,freeze_till=None,
                                            learning_rate=self.config.params_learning_rate)
        self.save_model(path=self.config.updated_basemodel,model=self.full_model)
    @staticmethod
    def save_model(path: Path,model: tf.keras.Model):
        model.save(path)


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [5]:
try:
    config = ConfigurationManager()
    prepareBaseModelConfig=config.get_PrepareBaseModelConfig()
    prepareBaseModel=PrepareBaseModel(config=prepareBaseModelConfig)
    prepareBaseModel.get_basemodel()
    prepareBaseModel.update_basemodel()

except Exception as e:
    raise e


[2026-05-17 06:55:22,829:INFO:common:yaml file:config\config.yamlloaded successfully]
[2026-05-17 06:55:22,834:INFO:common:yaml file:params.yamlloaded successfully]
[2026-05-17 06:55:22,835:INFO:common:created directory at:artifacts]
[2026-05-17 06:55:22,836:INFO:common:created directory at:artifacts/PrepareBaseModel]
[2026-05-17 06:55:23,823:WARNING:saving_utils:Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]


c:\Users\prasa\chicken-disease\chicken\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0     